In [ ]:
# Importation des outils nécessaire
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import ipywidgets as widgets
from ipywidgets import interact, interact, fixed, interact_manual
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.feature_selection import RFE

In [ ]:
# Importer des données

df = pd.read_csv('/content/Telco-Customer-Churn.txt')
df.head()

In [ ]:
# Information sur l'ensemble des données
df.info()

In [ ]:
# Nombre de valeurs uniques par colonne
df.nunique()

## Analyse exploratoire des données

In [ ]:
from pandas.core.arrays import categorical
# Variables catégorielles
categorical_columns = df.nunique()[df.nunique() < 5].keys().to_list()
categorical_columns

In [ ]:
# Création d'une fonction de construction de graphiques à barres et de maniére intéractive

def bar_plot(a):
  df[a].value_counts(normalize=True).plot(kind='bar')
  plt.ylabel('Proportion')
  plt.title('Distribution of ' + str(a))
  return plt.show()

In [ ]:
# Interactive

interact(bar_plot, a=categorical_columns)

# En séléctionnant Churn, on constate qu'il y'a plus de client qui ne sont pas désabonnés(plus de 70%)
# que de client qui se sont désabonné (moins de 30%).
# Donc nous avons un probléme de déséquilibre qu'il faudra absolument résoudre
# car cela peut impacter négativement la pérformance des modéles de classification que nous allons construire
# La distribution des données entre les différents sexes(gender) est à peu près également répartie
# Le nombre de clients (SeniorCitizen) qui sont des personnes âgés (moins de 20%) est bien inférieur à celui des clients jeunes (plus de 80%)
# La variable Partner indique si le client a un partenaire (Yes) ou pas (No). Les deux catégories ont pratiquement la même proportion



In [ ]:
# Echantillonnement aléatoire de la variable 'TotalCharge'
df['TotalCharges'].sample(10)

In [ ]:
# Variable quantitative
numerical_columns = df.nunique()[df.nunique() > 5].keys().to_list()
numerical_columns

In [ ]:
numerical_columns = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Création d'une fonction de construction d'histogramme et de maniére interactive

def hist_plot(b):
  sns.displot(df[b], kde= False)
  plt.title(f'Histogramme de {str(b)}')
  return plt.show()

# Convertir du type des valeurs de la variable 'TotalCharges' en float (décimal)
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype(float)

In [ ]:
# Interactive

interact(hist_plot, b=numerical_columns);

In [ ]:
# Coefficient d'asymetrie de la variable 'TotalCharges'

float(df['TotalCharges'].skew())

In [ ]:
# Création d'une fonction de construction de boites à moustaches et de maniére interactive

def box_plot(c):
  sns.boxplot(df[c])
  plt.title(f'Boite à moustaches de {str(c)}')
  return plt.show()

In [ ]:
# Interactive

interact(box_plot, c=numerical_columns);

In [ ]:
# TotalCharge vs Churn

sns.boxplot(
    x='Churn',
    y='TotalCharges',
    data=df,
    palette='tab10'   # ou 'pastel', 'coolwarm', 'Set1','Set2','Set3','Pastel1','Pastel2','Dark2','Accent','Paired','Tab10','Tab20'
)

plt.ylabel('Facture totale')
plt.xlabel('Désabonnement')
plt.title('Montant total de la facture client')
plt.show()


In [ ]:
# MonthlyCharges vs Churn


sns.boxplot(
    x='Churn',
    y='MonthlyCharges',
    data=df,
    palette='Set2'   # ou 'pastel', 'coolwarm', 'Set1','Set2','Set3','Pastel1','Pastel2','Dark2','Accent','Paired','Tab10','Tab20'
)

plt.ylabel('Facture mensuelle')
plt.xlabel('Désabonnement')
plt.title('Montant facturer mensuellement au client')
plt.show()


Le montant total facturé aux clients qui ont résilié leur contrat est inférieur au montant total facturé aux clients qui ne l'ont pas fais. Mais, les clients qui se sont désabonnés sont plus facturés mensuellement que les clients qui se sont pas désabonnés. Cette information est trés importante pour l'entreprise car le montant facturé mensuellement peut être un facteur important qui détermine si un client va se désabonner ou non.

In [ ]:
# tenure vs Churn

sns.boxplot(
    x='Churn',
    y='tenure',
    data=df,
    palette='tab10'   # ou 'pastel', 'coolwarm', 'Set1','Set2','Set3','Pastel1','Pastel2','Dark2','Accent','Paired','Tab10','Tab20'
)

plt.ylabel('Facture totale')
plt.xlabel('Désabonnement')
plt.title("Nombre de mois pendant lesquels le client est resté dans l'entreprise")
plt.show()


In [ ]:
# MonthlyCharges vs Churn by SeniorCitizen

sns.boxplot(
    x='Churn',
    y='MonthlyCharges',
    hue='SeniorCitizen',
    data=df,
    palette='tab10');

## Il semble que les personnes âgées soient beaucoup plus facturées que les jeunes. Malgrés cela ils restent plus fidéles à l'entreprise que les jeunes.

In [ ]:
# MonthlyCharges vs Churn by Dependents


sns.boxplot(
    x='Churn',
    y='MonthlyCharges',
    hue='Dependents',
    data=df,
    palette='tab10');

In [ ]:
# Résumé statistique
df.describe()

## Comme vous l'avez remarqué, les trois variables quantitatives (tenure, MonthlyCharges et TotalCharges) ont différentes échelles tenure varie entre 0 et 72 tandis que MonthlyCharges varie entre 18.25 et 118.75 et TotalCharges varie entre 18.8 et 8684.8. De nombreux modèles de Machine et de Deep Learning fonctinnent mieux avec des variables standardisées ou normalisées.

# On passe à la standardisarion des données dans la section de pretraitement des données

## Prétraitement des données

In [ ]:
# Gestion des valeurs manquants

data = df.copy()
data.isnull().sum()

In [ ]:
# Suppression des valeurs manquants

data.dropna(inplace=True)

In [ ]:
# Verification

data.isnull().sum()

In [ ]:
# Encodage des variables binaires
# En effet les algorithme de machine learning surtout ceux écrit dans la librairie scikit-learn ne prennent pas des valeurs chaines de caractére,
# elles ont bessoin uniquement les valeurs numérique

data['gender'] = data['gender'].apply(lambda x: 1 if x == 'Male' else 0)


In [ ]:

# Enumerer la liste des variables binaire à encoder le reste de mes variable
binary_columns = data.drop('gender', axis=1).nunique()[data.drop('gender', axis=1).nunique() < 3].keys().to_list()
binary_columns


In [ ]:
# Encoder ces variable binaire
for column in binary_columns:
  data[column] = data[column].apply(lambda x: 1 if x == 'Yes' else 0)

In [ ]:
# Encodage des variables catégorielles réstantes en excluant les variables binaires

remaining_cat_vars = data[categorical_columns].nunique()[data[categorical_columns].nunique() > 2].keys().to_list()
remaining_cat_vars_dummies = pd.get_dummies(data=data[remaining_cat_vars], columns=remaining_cat_vars, drop_first=True, dtype=int)

In [ ]:
# Nouvelle dataframe
data = pd.concat([data['gender'], data[binary_columns], remaining_cat_vars_dummies, data[numerical_columns]], axis=1)


In [ ]:
# Afficher la nouvelle data
data.head()

In [ ]:
# La dimension
data.shape

In [ ]:
# Transformation de la variable 'TotalCharge'
# En effet plusieur algorithme de machine learning sont conçus pour fonctionner avec des variable normale ou pseudo normale ou à distribution symétrique
# Or la variable 'TotalCharge' est trés asymétrique car le coefficient d'asymétri est trés proche de 1 (0.96)

data['TotalCharges'] = np.sqrt(data['TotalCharges'])

In [ ]:
# Histogramme de la variable transformée
sns.displot(data['TotalCharges'], kde=False)

In [ ]:
# Calculer à nouveau le coefficient d'asymétrie de la variable 'TotalCharges' après transformation
float(data['TotalCharges'].skew())

In [ ]:
# Division des données en trois
# Données d'entrainement (60%), de validation (20%), et de test (20%)

x = data.drop('Churn', axis=1)
y = data['Churn']

seed = 111 # Pour s'assurer la reproductibilité des résultats

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.4, random_state=seed, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, test_size=0.5, random_state=seed, stratify = y_test)

In [ ]:
# Fréquence des classes dans y

pd.DataFrame(y).value_counts(normalize = True)

In [ ]:
# Fréquence des classes dans y_train

pd.DataFrame(y_train).value_counts(normalize = True)

In [ ]:
# Fréquence des classes dans y_val

pd.DataFrame(y_val).value_counts(normalize = True)

In [ ]:
# Fréquence des classes dans y_test

pd.DataFrame(y_test).value_counts(normalize = True)

Attaquons-nous maintenant au problème de déséquilibre de classe dans les données.
Lorsqu'il y a une trés grande difference entre le nombre d'observations dans chaque catégories de la variable cuble à prédire, cela peut entrainer des érreurs de modélisations.

Dans notre cas ici, il y a un peu plus de 73% de personne qui n'ont pas résillié leur abonnenment contre environ un peu plus 26% qui ont résillié leur abonnement. Il y a donc un grand déséquilibre de classe. Nous pouvons utiliser le rééchantillonnage pour créer plus d'équilibre entre les catégories de la variable cible. Soit on crée plus d'observations dans la classe minoritaire (modalité 1) c'est-à-dire- on fait un sur-échantillonnage, soit on diminue les observations de la classe majoritaire (modalité 0) c'est-à-dire un sous-échantillonnage.

In [ ]:
# Résolution du problème de déséquilibre de classe : Méthode de sur-échantillonnage de la classe minoritaire

x2 = x_train
x2['Churn'] = y_train.values
minority = x2[x2.Churn == 1]
majority = x2[x2.Churn == 0]

minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=seed)
upsampled = pd.concat([majority, minority_upsampled])

upsampled

In [ ]:
# Vérification

upsampled['Churn'].value_counts(normalize = True)

In [ ]:
# Données d'entrainement sur la base la méthode de sur-échantillonnage de la classe minoritaire

x_train_up = upsampled.drop('Churn', axis=1)
y_train_up = upsampled['Churn']

In [ ]:
# Résolution du problème de déséquilibre de classe : Méthode de sous-échantillonnage de la classe majoritaire

majority_downsampled = resample(majority, replace=False, n_samples=len(minority), random_state=seed)
downsampled = pd.concat([minority, majority_downsampled])

downsampled

In [ ]:
# Vérification

downsampled['Churn'].value_counts(normalize = True)

In [ ]:
# Données d'entrainement sur la base la méthode de sous-échantillonnage de la classe majoritaire

x_train_down = downsampled.drop('Churn', axis=1)
y_train_down = downsampled['Churn']

In [ ]:
# y_train et x_train

y_train = x_train['Churn']

x_train = x_train.drop('Churn', axis=1)

In [ ]:
# Définition des données d'entrainement
# Choix possibles : (x_train_up, y_train_up) et (x_train_down, y_train_down)

train_feature = x_train_up
train_labels = y_train_up

## Passons finalement à la normalisation des données

In [ ]:
train_feature.head()

In [ ]:
# Normalisation des variables indépendantes des différents ensembles de données puisqu'on
# a des variables qui ont de plus grandes valeus et d'autre ne le sont pas comme la variable 'tenure', 'MonthlyCharges', et 'TotalCharges'

scaler = MinMaxScaler()    # Standardisation : scaler = StandardScaler()
mod_scaler = scaler.fit(train_feature)
train_feature = mod_scaler.transform(train_feature)
x_val = mod_scaler.transform(x_val)
x_test = mod_scaler.transform(x_test)

# Retransformation en Dataframe

train_feature = pd.DataFrame(train_feature, columns = x.columns)
x_val = pd.DataFrame(x_val, columns = x.columns)
x_test = pd.DataFrame(x_test, columns = x.columns)



In [ ]:
# Toutes les valeurs sont maintenant entre 0 et 1

train_feature.head()

In [ ]:
# Résumé statistique

train_feature.describe()

## Modélisation

In [ ]:
# Séléctionner des meilleures variables prédictives

rf = RandomForestClassifier()

rf.fit(train_feature, train_labels)

print(classification_report(y_val, rf.predict(x_val)))

In [ ]:
# Importer des variables importantes

vars_imp = pd.Series(rf.feature_importances_, index=train_feature.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
sns.barplot(x=vars_imp.index, y=vars_imp)
plt.xticks(rotation=90)
plt.xlabel("Variables")
plt.ylabel("Score d'importance de la variable")
plt.title("Importance de la variables prédictives")
plt.show()

In [ ]:
# Affichage de vars_impo

vars_imp

In [ ]:
# Variables séléctionnées pour les algorithmes

seuil = 0.004
vars_selected = vars_imp[vars_imp > seuil].index.to_list()

train_feature = train_feature[vars_selected]
x_val = x_val[vars_selected]
x_test = x_test[vars_selected]

In [ ]:
# Nombre de variables prédictives

len(vars_selected)

In [ ]:
len(train_feature.columns)

##<font color="red"><b>Modèle de Regression logistique</b></font>

In [ ]:
from ast import Param
# Dictionnaire des hyperparamètres

param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 50, 100, 500]}

# Objet GridSearchCV

grid_logreg_class = GridSearchCV(estimator=LogisticRegression(random_state=seed,max_iter=500),param_grid=param_grid, scoring='f1', cv=5)

# Entrainement de l'algorithme

logreg_model = grid_logreg_class.fit(train_feature, train_labels)

# Meilleursscore et meilleur hyperparamètres

print(logreg_model.best_score_, 3)

print(logreg_model.best_estimator_)

## Le modèle a un bon score d'entrainement. Evaluons sa performance sur les données de validation afin d'apprécier sa capacité à générer sur de nouvelles données.

In [ ]:
# Fonction d'évaluation de la performance

def model_evaluation(model, features, labels):
  pred = model.predict(features)
  print(classification_report(labels, pred))

In [ ]:
# Evaluation du modèle de régression logistique

model_evaluation(logreg_model, x_val, y_val)

## Appliquons l'algorithme Recurcive Feature Eliminator (RFE) sur le modèle afin de voir s'il garde les mêmes performances lorqu'on réduit le nombre de prédicteurs. En effet, plus le modèle est complexe, plus il est dificile de l'interpreter.

##<font color="red"><b>Fonction de construction d'un modèle avec utilisation de l'algorithme RFE</b></font>

In [ ]:
# Création d'une fonction de construction d'un modèle avec utilisation de l'algorithme RFE

def model_with_rfe(model):
  rfe_model = RFE(estimator=model, verbose=0)
  rfe_model.fit(train_feature, train_labels)
  mask = rfe_model.support_
  reduced_x = train_feature.loc[:, mask]
  print(reduced_x.columns)
  return rfe_model

In [ ]:
# Logistic Regression RFE

rfe_logreg_model = model_with_rfe(logreg_model.best_estimator_)

rfe_logreg_model

In [ ]:
# Evaluation du model de régression logistique avec RFE

model_evaluation(rfe_logreg_model, x_val, y_val)

# Le RFE a réduit le nombre de prédictive de 28 à 14 et n'a pas améliorer la performance du modèle. Cela montre non seulement la puissance de cet algorithme mais aussi l'importance de la séléction des meilleurs prédictives en modélisation.

##<font color="red"><b>Modèle de Forêt Aléatoire</b></font>

In [ ]:
# Dictionnaire des hypérparametres

param_grid_rf = {'n_estimators': [10, 50, 100, 1000],
              'max_depth': [3, 5, 10, 20, None]}

# Objet GridSearchCV

grid_rf_class = GridSearchCV(estimator=RandomForestClassifier(random_state=seed),
                                 param_grid=param_grid_rf,
                                 scoring='f1',
                                 cv=5)

# Entrainement de l'algorithme

rf_model = grid_rf_class.fit(train_feature, train_labels)

# Meilleur score et meilleur hyperparamètres

print(round(rf_model.best_score_, 3))

print(rf_model.best_estimator_)

In [ ]:
# Evalution du modèle de forêt aléatoire

model_evaluation(rf_model.best_estimator_, x_val, y_val)

## Comparé aux valeurs du modèle de régression logistique, le modèle de forêt aléatoire semble moins efficace.

##<font color="red"><b>Appliquons une RFE</b></font>

In [ ]:
# Random Forest avec RFE

rfe_forest_model = model_with_rfe(rf_model.best_estimator_)

rfe_forest_model

In [ ]:
# Evaluation du modèle de forêt aléatoire avec RFE

model_evaluation(rfe_forest_model, x_val, y_val)

## Au vu de ces valeurs, nous retiendrons le modèle de forêt aléatoire abtenu sans RFE.

## Voyons maintenant à la construction d'un modèle de réseau de neuronnes artificiel.

##<font color="red"><b>Classification Perception multicouche</b></font>


In [ ]:
# MLPClassifier

mlp = MLPClassifier(random_state=seed, max_iter=1000)

parameters = {'hidden_layer_sizes' :[(50,), (100,), (200,)],
              #'activation' :['identity', 'logistic', 'tanh', 'relu'],
              'learning_rate' :['constant', 'invscaling', 'adaptive']}

mlp_cv = GridSearchCV(mlp, parameters, scoring='f1', cv=5, n_jobs=1)

# Entrainer l'algorithme

mlp_cv.fit(train_feature, train_labels)

In [ ]:
# Meilleur score et meilleur hyperparamètre

float(round(mlp_cv.best_score_, 3))

In [ ]:
mlp_cv.best_estimator_

In [ ]:
# Evaluation du modèle perception

model_evaluation(mlp_cv.best_estimator_, x_val, y_val)

## Passons maintenant à un modèle SVM (Support Vector Machine)

##<font color="red"><b>Support Vector Machine</b></font>


In [ ]:
# Support Vector Machine : Classifier qui trouve l'hyperplan optimal qui maximise la frontière entre 2 classes

svm_model = SVC(random_state=seed)

svm_hyp = {'kernel' :['linear', 'rbf'], 'C' :[0.1, 1.0, 10, 50, 100]}

svm_cv = GridSearchCV(svm_model, svm_hyp, scoring='f1', cv=5)

# Entrainer l'algorithme

svm_cv.fit(train_feature, train_labels)

# Meilleur score et meilleur hyperparametre

print(round(svm_cv.best_score_, 3))

print(svm_cv.best_estimator_)

In [ ]:
# Evaluation du modèle SVM

model_evaluation(svm_cv.best_estimator_, x_val, y_val)

##<font color="green"><b>Conclusion :</b> </font>


## J'ai utilisés les données d'évaluation pour sélectionner le meilleur modèle. Ensuite nous évaluons le meilleur modèle sélectionné sur les données de test afin d'apprécier sa performance sur de nouvelle données. Idéalement les performances de ce modèle sur les données d'évaluation et sur les données de test doivent être relativement proches.

In [ ]:
# Sur les donnés de validation
model_evaluation(logreg_model.best_estimator_, x_val, y_val)

In [ ]:
# Performance du meilleur modèle sur les données de test

model_evaluation(logreg_model.best_estimator_, x_test, y_test)